In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ---------------------------------------------------------
# 1. 파일 로드 및 설정
# ---------------------------------------------------------
# 파일들이 있는 현재 디렉터리 경로 (필요시 수정)
base_path = "." 

# 파일 목록 정의
files = {
    "discharge": "discharge.csv",
    "discharge_detail": "discharge_detail.csv",
    "radiology": "radiology.csv",
    "radiology_detail": "radiology_detail.csv"
}

# 그래프 스타일 설정 (한글 폰트 깨짐 방지 - Windows 기준)
plt.rc('font', family='Malgun Gothic') 
plt.rcParams['axes.unicode_minus'] = False

# 데이터프레임을 저장할 딕셔너리
dfs = {}

print(">>> 데이터 로드를 시작합니다...")

for key, filename in files.items():
    file_path = os.path.join(base_path, filename)
    
    if os.path.exists(file_path):
        try:
            # 데이터 로드 (파일이 클 경우를 대비해 low_memory=False 설정)
            dfs[key] = pd.read_csv(file_path, low_memory=False)
            print(f"[성공] {filename} 로드 완료! (행: {dfs[key].shape[0]}, 열: {dfs[key].shape[1]})")
        except Exception as e:
            print(f"[에러] {filename} 로드 중 오류 발생: {e}")
    else:
        print(f"[경고] {filename} 파일이 경로에 존재하지 않습니다.")

print("-" * 50)

# ---------------------------------------------------------
# 2. 데이터 기본 정보 확인 및 EDA 함수 정의
# ---------------------------------------------------------
def perform_eda(df, name):
    print(f"\nStart EDA for: [ {name} ]")
    print("=" * 40)
    
    # 2.1 기본 정보 (컬럼, 데이터 타입, 결측치)
    print("\n1. 데이터 기본 정보 (Info):")
    print(df.info())
    
    # 2.2 결측치 확인
    print("\n2. 결측치 개수 (Missing Values):")
    missing = df.isnull().sum()
    print(missing[missing > 0])
    
    # 2.3 고유 환자 및 입원 건수 확인 (컬럼이 존재할 경우)
    print("\n3. 고유 ID 카운트 (Unique Counts):")
    if 'subject_id' in df.columns:
        print(f" - 고유 환자 수 (subject_id): {df['subject_id'].nunique():,}")
    if 'hadm_id' in df.columns:
        print(f" - 고유 입원 수 (hadm_id): {df['hadm_id'].nunique():,}")
        
    # 2.4 텍스트 데이터 분석 (text 컬럼이 있는 경우)
    if 'text' in df.columns:
        print("\n4. 텍스트 데이터 분석:")
        # 텍스트 길이 계산 (글자 수)
        df['text_len'] = df['text'].astype(str).apply(len)
        
        print(f" - 평균 글자 수: {df['text_len'].mean():.2f}")
        print(f" - 최대 글자 수: {df['text_len'].max()}")
        print(f" - 최소 글자 수: {df['text_len'].min()}")
        
        # 텍스트 길이 분포 시각화
        plt.figure(figsize=(10, 5))
        sns.histplot(df['text_len'], bins=50, kde=True)
        plt.title(f"{name} - Text Length Distribution")
        plt.xlabel("Text Length (Characters)")
        plt.ylabel("Count")
        plt.show()
        
        # 샘플 텍스트 출력 (첫 번째 데이터)
        print("\n[샘플 텍스트 (상위 1개)]")
        print("-" * 30)
        print(df['text'].iloc[0][:500] + "...\n(후략)") # 너무 기니까 앞 500자만 출력
        print("-" * 30)
    
    else:
        print("\n4. 텍스트('text') 컬럼이 없습니다. 메타데이터 파일로 추정됩니다.")
        # 수치형 데이터가 있다면 기초통계량 출력
        print(df.describe())

    print("\n" + "=" * 40 + "\n")

# ---------------------------------------------------------
# 3. 각 파일별 EDA 실행
# ---------------------------------------------------------
for key, df in dfs.items():
    perform_eda(df, key)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import re

# 1. 데이터 준비 (예시)
df_disch = dfs['discharge'].copy()

# 2. 텍스트 전처리 함수 (특수문자 제거, 소문자 변환)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\[\*\*.*?\*\*\]', '', text) # MIMIC의 익명화 마크([**...**]) 제거
    text = re.sub(r'[^a-z\s]', ' ', text)       # 알파벳과 공백만 남김
    text = re.sub(r'\s+', ' ', text).strip()    # 다중 공백 제거
    return text

print("텍스트 전처리 중...")
df_disch['clean_text'] = df_disch['text'].apply(clean_text)

# ---------------------------------------------------------
# 방법 A: TF-IDF 벡터화 (자동으로 중요 단어 수치화)
# ---------------------------------------------------------
print("TF-IDF 변환 중...")
# max_features=50: 가장 중요한 단어 50개만 뽑습니다 (과적합 방지 및 계산 속도 위해)
tfidf = TfidfVectorizer(max_features=50, stop_words='english') 
tfidf_matrix = tfidf.fit_transform(df_disch['clean_text'])

# 결과를 데이터프레임으로 변환
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(), 
    columns=[f"tfidf_{word}" for word in tfidf.get_feature_names_out()]
)

# 원본 데이터와 합치기 (hadm_id 기준)
final_df_tfidf = pd.concat([df_disch[['hadm_id']], tfidf_df], axis=1)

# ---------------------------------------------------------
# 방법 B: 주요 키워드 카운팅 (Rule-based)
# ---------------------------------------------------------
keywords = ['fail', 'pain', 'fever', 'infection', 'bleed', 'unstable']

print("키워드 카운팅 중...")
for word in keywords:
    # 텍스트 내 해당 단어 등장 횟수 계산
    df_disch[f'count_{word}'] = df_disch['clean_text'].apply(lambda x: x.count(word))

# 필요한 컬럼만 선택
final_df_counts = df_disch[['hadm_id'] + [f'count_{word}' for word in keywords]]

# ---------------------------------------------------------
# 결과 확인
# ---------------------------------------------------------
print("\n[TF-IDF 결과 샘플]")
print(final_df_tfidf.head(3))

print("\n[키워드 카운팅 결과 샘플]")
print(final_df_counts.head(3))

In [ ]:
import pandas as pd
import os

# ---------------------------------------------------------
# 1. 파일 로드
# ---------------------------------------------------------
# 파일 경로 설정 (admissions.csv 파일이 있는 경로로 수정하세요)
file_path = 'data/physionet.org/files/mimiciv/3.1/hosp/admissions.csv' 

if os.path.exists(file_path):
    # 날짜 컬럼을 인식하며 로드 (속도 향상 및 바로 연산 가능)
    df_adm = pd.read_csv(file_path, parse_dates=['admittime', 'dischtime', 'deathtime'])
    print(f"데이터 로드 완료: {df_adm.shape}")
else:
    print(f"파일을 찾을 수 없습니다: {file_path}")
    # (테스트를 위한 더미 데이터 생성 코드는 생략합니다. 실제 파일을 사용하세요)

# ---------------------------------------------------------
# 2. 데이터 정렬 (가장 중요!)
# ---------------------------------------------------------
# 환자별(subject_id)로 묶고, 시간순서(admittime)대로 정렬해야 
# '다음 입원'이 무엇인지 알 수 있습니다.
df_adm = df_adm.sort_values(by=['subject_id', 'admittime'])

# ---------------------------------------------------------
# 3. '다음 입원 시각' 가져오기 (Shift 기능 활용)
# ---------------------------------------------------------
# groupby().shift(-1): 같은 환자 그룹 내에서 '바로 다음 행'의 값을 가져와 현재 행 옆에 붙입니다.
df_adm['next_admittime'] = df_adm.groupby('subject_id')['admittime'].shift(-1)
df_adm['next_hadm_id'] = df_adm.groupby('subject_id')['hadm_id'].shift(-1) # 확인용

# ---------------------------------------------------------
# 4. 시간 차이 계산 (다음 입원일 - 현재 퇴원일)
# ---------------------------------------------------------
# 결과는 Timedelta 타입 (일, 시, 분, 초 차이)이 됩니다.
df_adm['time_diff'] = df_adm['next_admittime'] - df_adm['dischtime']

# ---------------------------------------------------------
# 5. 라벨링: 30일 이내 재입원 여부 (Target Y 생성)
# ---------------------------------------------------------
# 조건 1: time_diff가 존재해야 함 (마지막 입원이 아님)
# 조건 2: time_diff가 30일 이내여야 함 (<= 30 days)
# 조건 3: time_diff가 0보다 커야 함 (데이터 오류나 겹치는 입원 방지)

def check_readmission(diff):
    if pd.isnull(diff): # 다음 입원이 없으면 재입원 아님
        return 0
    elif 0 <= diff.days <= 30: # 30일 이내 재입원
        return 1
    else: # 30일 넘어 재입원했거나(31일~), 값이 이상한 경우
        return 0

df_adm['readmission_30d'] = df_adm['time_diff'].apply(check_readmission)

# ---------------------------------------------------------
# 6. 선택 사항: 사망 환자 처리
# ---------------------------------------------------------
# 병원 내에서 사망한 경우(hospital_expire_flag == 1)는 물리적으로 재입원이 불가능하므로 
# 라벨이 0이어야 정상입니다. (위 로직에서도 다음 입원이 없으니 0이 나오겠지만 확실하게 처리)
if 'hospital_expire_flag' in df_adm.columns:
    df_adm.loc[df_adm['hospital_expire_flag'] == 1, 'readmission_30d'] = 0

# ---------------------------------------------------------
# 결과 확인 및 저장
# ---------------------------------------------------------
print("\n[라벨링 결과 요약]")
print(df_adm['readmission_30d'].value_counts())

# 검증을 위해 눈으로 확인 (재입원한 케이스만 일부 출력)
print("\n[재입원 케이스 샘플 (상위 5개)]")
cols_to_view = ['subject_id', 'hadm_id', 'dischtime', 'next_admittime', 'time_diff', 'readmission_30d']
print(df_adm[df_adm['readmission_30d'] == 1][cols_to_view].head(5))

# 필요한 경우 CSV로 저장
# df_adm.to_csv('admissions_with_label.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# ---------------------------------------------------------
# 1. 데이터 병합 (Merge)
# ---------------------------------------------------------
# df_adm: 타겟(Y)이 있는 데이터
# final_df_tfidf: 입력(X)이 있는 데이터 (TF-IDF 피처)

print(">>> 데이터 병합 시작...")

# 병합 기준: hadm_id (입원 ID)
# how='inner': 타겟과 텍스트 데이터가 모두 존재하는 입원 건만 사용
model_df = pd.merge(
    df_adm[['hadm_id', 'readmission_30d']], 
    final_df_tfidf, 
    on='hadm_id', 
    how='inner'
)

print(f"병합 완료! 데이터 크기: {model_df.shape}")
print(f"레이블 분포:\n{model_df['readmission_30d'].value_counts()}")

# ---------------------------------------------------------
# 2. 데이터셋 분리 (Train / Test Split)
# ---------------------------------------------------------
# 입력 변수(X): hadm_id와 타겟 변수를 제외한 모든 컬럼 (TF-IDF 단어들)
X = model_df.drop(['hadm_id', 'readmission_30d'], axis=1)

# 타겟 변수(Y): 재입원 여부
y = model_df['readmission_30d']

# 훈련셋:테스트셋 = 8:2 분할
# stratify=y: 재입원(1)과 비재입원(0)의 비율을 유지하며 분할 (불균형 데이터에 필수)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set 크기: {X_train.shape}")
print(f"Test set 크기:  {X_test.shape}")

# ---------------------------------------------------------
# 3. 모델 학습 (Random Forest)
# ---------------------------------------------------------
print("\n>>> 모델 학습 시작 (Random Forest)...")

# class_weight='balanced': 재입원 환자(소수 클래스)에 더 가중치를 둬서 학습
rf_model = RandomForestClassifier(
    n_estimators=100,       # 트리 개수
    max_depth=10,           # 트리 깊이 제한 (과적합 방지)
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1               # 모든 CPU 코어 사용
)

rf_model.fit(X_train, y_train)
print("학습 완료!")

# ---------------------------------------------------------
# 4. 성능 평가
# ---------------------------------------------------------
# 예측 수행
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1] # 재입원일 확률 (ROC-AUC용)

print("\n" + "="*50)
print("   [ 모델 평가 결과 Report ]")
print("="*50)

# (1) 분류 리포트 (Precision, Recall, F1-score)
print(classification_report(y_test, y_pred))

# (2) ROC-AUC 점수
roc_score = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_score:.4f}")

# ---------------------------------------------------------
# 5. 시각화 (Confusion Matrix & Feature Importance)
# ---------------------------------------------------------
plt.figure(figsize=(12, 5))

# (1) Confusion Matrix (오차 행렬)
plt.subplot(1, 2, 1)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0.5, 1.5], ['Non-Readmit', 'Readmit'])
plt.yticks([0.5, 1.5], ['Non-Readmit', 'Readmit'])

# (2) Feature Importance (중요 단어 확인)
plt.subplot(1, 2, 2)
importances = rf_model.feature_importances_
feature_names = X.columns
# 상위 10개 중요 피처 추출
indices = np.argsort(importances)[::-1][:10]

plt.title("Top 10 Important Words (Features)")
plt.barh(range(10), importances[indices], align='center')
plt.yticks(range(10), [feature_names[i] for i in indices])
plt.gca().invert_yaxis() # 상위 중요도가 위로 오게 뒤집기
plt.xlabel("Relative Importance")

plt.tight_layout()
plt.show()

## 성능 향상을 위해 전처리 진행 후 모델 학습 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import re

# 1. 데이터 준비 (예시)
df_disch = dfs['discharge'].copy()

# 2. 텍스트 전처리 함수 (특수문자 제거, 소문자 변환)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\[\*\*.*?\*\*\]', '', text) # MIMIC의 익명화 마크([**...**]) 제거
    text = re.sub(r'[^a-z\s]', ' ', text)       # 알파벳과 공백만 남김
    text = re.sub(r'\s+', ' ', text).strip()    # 다중 공백 제거
    return text

print("텍스트 전처리 중...")
df_disch['clean_text'] = df_disch['text'].apply(clean_text)

# ---------------------------------------------------------
# 방법 A: TF-IDF 벡터화 (자동으로 중요 단어 수치화)
# ---------------------------------------------------------
print("TF-IDF 변환 중...")
# max_features=50: 가장 중요한 단어 50개만 뽑습니다 (과적합 방지 및 계산 속도 위해)
tfidf = TfidfVectorizer(max_features=50, stop_words='english') 
tfidf_matrix = tfidf.fit_transform(df_disch['clean_text'])

# 결과를 데이터프레임으로 변환
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(), 
    columns=[f"tfidf_{word}" for word in tfidf.get_feature_names_out()]
)

# 원본 데이터와 합치기 (hadm_id 기준)
final_df_tfidf = pd.concat([df_disch[['hadm_id']], tfidf_df], axis=1)

# ---------------------------------------------------------
# 방법 B: 주요 키워드 카운팅 (Rule-based)
# ---------------------------------------------------------
keywords = ['fail', 'pain', 'fever', 'infection', 'bleed', 'unstable']

print("키워드 카운팅 중...")
for word in keywords:
    # 텍스트 내 해당 단어 등장 횟수 계산
    df_disch[f'count_{word}'] = df_disch['clean_text'].apply(lambda x: x.count(word))
import re
import pandas as pd

# 1. 섹션 추출 함수 정의
def extract_section(text, section_name):
    """
    text: 전체 퇴원 요약지 텍스트
    section_name: 찾고자 하는 섹션 헤더 (예: "Hospital Course")
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # 정규표현식 패턴: 
    # 1) section_name으로 시작하고 (: 포함)
    # 2) 그 뒤의 내용((.*?))을 캡처하되
    # 3) 다음 섹션 헤더(대문자로 시작하고 :로 끝나는 줄)가 나오기 전까지
    pattern = rf"{section_name}:?\s*(.*?)(?=\n[A-Z][A-Za-z ]+:|$)"
    
    match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
    
    if match:
        return match.group(1).strip()
    else:
        return "" # 해당 섹션이 없으면 빈 문자열 반환

# 2. 실제 데이터에 적용 (df_disch 데이터프레임이 있다고 가정)
print(">>> 주요 섹션 추출 중...")

# 가장 중요한 'Brief Hospital Course' (입원 경과)만 추출
# (MIMIC 데이터에는 'Brief Hospital Course' 또는 그냥 'Hospital Course'로 적혀있음)
df_disch['hospital_course'] = df_disch['text'].apply(
    lambda x: extract_section(x, "Hospital Course")
)

# 만약 Hospital Course가 비어있으면 'History of Present Illness'라도 가져오게 하기 (보완)
mask_empty = df_disch['hospital_course'] == ""
df_disch.loc[mask_empty, 'hospital_course'] = df_disch.loc[mask_empty, 'text'].apply(
    lambda x: extract_section(x, "History of Present Illness")
)

# 3. 결과 확인
print("추출 완료!")
print(f"- 원본 텍스트 평균 길이: {df_disch['text'].str.len().mean():.0f}")
print(f"- 추출된 섹션 평균 길이: {df_disch['hospital_course'].str.len().mean():.0f}")

# 샘플 출력
print("\n[추출된 Hospital Course 샘플]")
print(df_disch['hospital_course'].iloc[0][:300] + "...")

# 필요한 컬럼만 선택
final_df_counts = df_disch[['hadm_id'] + [f'count_{word}' for word in keywords]]



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import re

# 1. 데이터 준비 (예시)
df_disch = dfs['discharge'].copy()

# 2. 텍스트 전처리 함수 (특수문자 제거, 소문자 변환)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\[\*\*.*?\*\*\]', '', text) # MIMIC의 익명화 마크([**...**]) 제거
    text = re.sub(r'[^a-z\s]', ' ', text)       # 알파벳과 공백만 남김
    text = re.sub(r'\s+', ' ', text).strip()    # 다중 공백 제거
    return text

print("텍스트 전처리 중...")
df_disch['clean_text'] = df_disch['text'].apply(clean_text)

# ---------------------------------------------------------
# 방법 A: TF-IDF 벡터화 (자동으로 중요 단어 수치화)
# ---------------------------------------------------------
print("TF-IDF 변환 중...")
# max_features=50: 가장 중요한 단어 50개만 뽑습니다 (과적합 방지 및 계산 속도 위해)
tfidf = TfidfVectorizer(max_features=50, stop_words='english') 
tfidf_matrix = tfidf.fit_transform(df_disch['clean_text'])

# 결과를 데이터프레임으로 변환
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(), 
    columns=[f"tfidf_{word}" for word in tfidf.get_feature_names_out()]
)

# 원본 데이터와 합치기 (hadm_id 기준)
final_df_tfidf = pd.concat([df_disch[['hadm_id']], tfidf_df], axis=1)

# ---------------------------------------------------------
# 방법 B: 주요 키워드 카운팅 (Rule-based)
# ---------------------------------------------------------
keywords = ['fail', 'pain', 'fever', 'infection', 'bleed', 'unstable']

print("키워드 카운팅 중...")
for word in keywords:
    # 텍스트 내 해당 단어 등장 횟수 계산
    df_disch[f'count_{word}'] = df_disch['clean_text'].apply(lambda x: x.count(word))

# 필요한 컬럼만 선택
final_df_counts = df_disch[['hadm_id'] + [f'count_{word}' for word in keywords]]

# ---------------------------------------------------------
# 결과 확인
# ---------------------------------------------------------
print("\n[TF-IDF 결과 샘플]")
print(final_df_tfidf.head(3))

print("\n[키워드 카운팅 결과 샘플]")
print(final_df_counts.head(3))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# ---------------------------------------------------------
# 1. 데이터 병합 (Merge)
# ---------------------------------------------------------
# df_adm: 타겟(Y)이 있는 데이터
# final_df_tfidf: 입력(X)이 있는 데이터 (TF-IDF 피처)

print(">>> 데이터 병합 시작...")

# 병합 기준: hadm_id (입원 ID)
# how='inner': 타겟과 텍스트 데이터가 모두 존재하는 입원 건만 사용
model_df = pd.merge(
    df_adm[['hadm_id', 'readmission_30d']], 
    final_df_tfidf, 
    on='hadm_id', 
    how='inner'
)

print(f"병합 완료! 데이터 크기: {model_df.shape}")
print(f"레이블 분포:\n{model_df['readmission_30d'].value_counts()}")

# ---------------------------------------------------------
# 2. 데이터셋 분리 (Train / Test Split)
# ---------------------------------------------------------
# 입력 변수(X): hadm_id와 타겟 변수를 제외한 모든 컬럼 (TF-IDF 단어들)
X = model_df.drop(['hadm_id', 'readmission_30d'], axis=1)

# 타겟 변수(Y): 재입원 여부
y = model_df['readmission_30d']

# 훈련셋:테스트셋 = 8:2 분할
# stratify=y: 재입원(1)과 비재입원(0)의 비율을 유지하며 분할 (불균형 데이터에 필수)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set 크기: {X_train.shape}")
print(f"Test set 크기:  {X_test.shape}")

# ---------------------------------------------------------
# 3. 모델 학습 (Random Forest)
# ---------------------------------------------------------
print("\n>>> 모델 학습 시작 (Random Forest)...")

# class_weight='balanced': 재입원 환자(소수 클래스)에 더 가중치를 둬서 학습
rf_model = RandomForestClassifier(
    n_estimators=100,       # 트리 개수
    max_depth=10,           # 트리 깊이 제한 (과적합 방지)
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1               # 모든 CPU 코어 사용
)

rf_model.fit(X_train, y_train)
print("학습 완료!")

# ---------------------------------------------------------
# 4. 성능 평가
# ---------------------------------------------------------
# 예측 수행
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1] # 재입원일 확률 (ROC-AUC용)

print("\n" + "="*50)
print("   [ 모델 평가 결과 Report ]")
print("="*50)

# (1) 분류 리포트 (Precision, Recall, F1-score)
print(classification_report(y_test, y_pred))

# (2) ROC-AUC 점수
roc_score = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_score:.4f}")

# ---------------------------------------------------------
# 5. 시각화 (Confusion Matrix & Feature Importance)
# ---------------------------------------------------------
plt.figure(figsize=(12, 5))

# (1) Confusion Matrix (오차 행렬)
plt.subplot(1, 2, 1)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0.5, 1.5], ['Non-Readmit', 'Readmit'])
plt.yticks([0.5, 1.5], ['Non-Readmit', 'Readmit'])

# (2) Feature Importance (중요 단어 확인)
plt.subplot(1, 2, 2)
importances = rf_model.feature_importances_
feature_names = X.columns
# 상위 10개 중요 피처 추출
indices = np.argsort(importances)[::-1][:10]

plt.title("Top 10 Important Words (Features)")
plt.barh(range(10), importances[indices], align='center')
plt.yticks(range(10), [feature_names[i] for i in indices])
plt.gca().invert_yaxis() # 상위 중요도가 위로 오게 뒤집기
plt.xlabel("Relative Importance")

plt.tight_layout()
plt.show()